In [14]:
from fastai.data.core import DataLoaders
from huggingface_hub import login
from datasets import load_dataset
from collections import Counter
import pandas as pd
from fastai.data.transforms import Transform
import numpy as np
import re
import torch
import torch.nn as nn
from gensim.models import KeyedVectors
import gensim.downloader as api
from torch.utils.data import TensorDataset
from fastai.learner import Learner
from fastai.metrics import accuracy

In [2]:
# login to huggingface
login("")

In [3]:
# load dataset
dataset = load_dataset('')

In [4]:
# access samples
print(dataset['train'][332])
print(dataset['train'][331])

{'item': 'http://www.wikidata.org/entity/Q252187', 'name': 'áo dài', 'description': 'Vietnamese national costume, tunic', 'type': 'concept', 'category': 'fashion', 'subcategory': 'clothing', 'label': 'cultural representative'}
{'item': 'http://www.wikidata.org/entity/Q4778419', 'name': 'Anzac Day clash', 'description': 'traditional Australian football match between Collingwood and Essendon held annually on Anzac Day', 'type': 'concept', 'category': 'sports', 'subcategory': 'recurring sporting event', 'label': 'cultural exclusive'}


In [5]:
# label distribution
print(f"total number of samples in train: {len(dataset['train'])}")
print(f"total number of samples in validation: {len(dataset['validation'])}")

dist = Counter(sample["label"] for sample in dataset["train"])
print(dist)

total number of samples in train: 6251
total number of samples in validation: 300
Counter({'cultural exclusive': 2691, 'cultural agnostic': 1872, 'cultural representative': 1688})


In [6]:
df = pd.DataFrame(dataset["train"])
df.iloc[0]

item           http://www.wikidata.org/entity/Q32786
name                                             916
description                  2012 film by M. Mohanan
type                                          entity
category                                       films
subcategory                                     film
label                             cultural exclusive
Name: 0, dtype: object

In [7]:
# create the transformer dataframe row -> dataset sample

class W2VTransform(Transform):
    def __init__(self, w2v_model, max_len=30):
        super().__init__()
        self.w2v = w2v_model
        self.max_len = max_len
        self.dim = self.w2v.vector_size

    def enc(self, text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        tokens = text.split()
        vectors = [self.w2v[t] for t in tokens if t in self.w2v]

        if not vectors:
            return np.zeros(self.dim)

        # Mean pooling
        return np.mean(vectors, axis=0)

    def extract_text(self, df_row):

        return f"{df_row['name']} {df_row['description']} {df_row['type']}"


    def encodes(self, df_row):
        """
        param: df_row a dataframe row
        """

        x = self.enc(self.extract_text(df_row))
        y = df_row["label_d"]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y)


In [8]:
df_train = pd.DataFrame(dataset["train"])
df_valid = pd.DataFrame(dataset["validation"])

LABEL_MAP = {
    'cultural agnostic': 0,
    'cultural representative': 1,
    'cultural exclusive': 2
}

# map digit labels
df_train['label_d'] = df_train['label'].map(LABEL_MAP)
df_valid['label_d'] = df_valid['label'].map(LABEL_MAP)

# load word2vec pretrained
w2v_model = api.load('word2vec-google-news-300')

# transformer instantiation
transformer = W2VTransform(w2v_model)

# create train samples
X_train = []
y_train = []
for _, row in df_train.iterrows():
    x_vec, label = transformer.encodes(row)
    X_train.append(x_vec)
    y_train.append(label)

X_train = torch.stack(X_train)
y_train = torch.tensor(y_train, dtype=torch.long)

# create validation samples
X_valid = []
y_valid = []
for _, row in df_valid.iterrows():
    x_vec, label = transformer.encodes(row)
    X_valid.append(x_vec)
    y_valid.append(label)

X_valid = torch.stack(X_valid)
y_valid = torch.tensor(y_valid, dtype=torch.long)


train_ds = TensorDataset(X_train, y_train)
valid_ds = TensorDataset(X_valid, y_valid)

dls = DataLoaders.from_dsets(train_ds, valid_ds, bs=32, shuffle_train=True)


In [17]:
# classifier model

class W2VClassifier(nn.Module):
    def __init__(self, input_dim, hidden=128, out_dim=3):
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

In [18]:
# train

model = W2VClassifier(input_dim=w2v_model.vector_size)

learner = Learner(dls, model, loss_func=torch.nn.CrossEntropyLoss(), metrics=accuracy)
print(type(learner))

learner.fit_one_cycle(num_epochs=10)

AttributeError: cannot assign module before Module.__init__() call

In [1]:
from fastai.learner import Learner
from fastai.data.core import DataLoaders
from fastai.metrics import accuracy
from torch.utils.data import TensorDataset
import torch
import torch.nn as nn

# Dummy data
X = torch.randn(100, 300)
y = torch.randint(0, 3, (100,))
train_ds = TensorDataset(X[:80], y[:80])
valid_ds = TensorDataset(X[80:], y[80:])
dls = DataLoaders.from_dsets(train_ds, valid_ds, bs=16)

# Simple model
class W2VClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(300, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )

    def forward(self, x): return self.layers(x)

model = W2VClassifier()
learn = Learner(dls, model, loss_func=nn.CrossEntropyLoss(), metrics=accuracy)
learn.fit_one_cycle(3, 1e-3)

AttributeError: 'W2VClassifier' object has no attribute 'fit_one_cycle'

In [2]:
import fastai, fastcore
print(fastai.__version__)     # should be >= 2.7
print(fastcore.__version__)   # should be >= 1.5

2.8.2
1.8.2
